In [1]:
"""
4. Movement Pruning — ResNet-50 / CIFAR-10
==========================================
Movement pruning (Sanh et al., 2020) scores weights by their
movement DURING training rather than their absolute magnitude.
Weights moving toward zero get pruned; weights moving away from
zero are kept — even if currently small.

Score(w) = -w * dL/dw  (movement = negative gradient × weight)
         = sign(w) * |change in w| approximation

Since we have a pre-trained frozen model (no fine-tuning here),
we simulate movement scores using gradient-based importance:
  Score = w * |gradient|  (Taylor expansion approximation)

This is computed on a calibration batch before pruning.

Output: 4_movement_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "4_v2_movement_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS    = [0.30] # , 0.50, 0.70, 0.80, 0.90
CALIBRATION_BATCHES = 10   # batches used to estimate movement scores
MAX_ACC_DROP        = 0.02
INFERENCE_RUNS      = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Movement Score Estimation ─────────────────────────────────────────────────
def compute_movement_scores(model, loader, n_batches=10):
    """
    Estimate movement scores using first-order Taylor approximation:
      Importance(w_i) = |w_i * grad_i|
    This approximates the change in loss when w_i is zeroed out.

    Returns: dict mapping (module_name, param_name) -> score tensor
    """
    model.train()  # enable gradients
    criterion = nn.CrossEntropyLoss()
    scores    = {}

    # Accumulate gradients over calibration batches
    model.zero_grad()
    n_processed = 0
    for batch_idx, (inputs, targets) in enumerate(loader):
        if batch_idx >= n_batches:
            break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)
        loss    = criterion(outputs, targets)
        loss.backward()
        n_processed += 1

    # Compute importance = |w * grad| for each Conv2d/Linear weight
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w    = module.weight.data
            grad = module.weight.grad
            if grad is not None:
                # Taylor importance: absolute weight × gradient
                score = (w * grad / n_processed).abs()
                scores[name] = score.detach().cpu()

    model.zero_grad()
    model.eval()
    return scores


def apply_movement_pruning(model, loader, sparsity, n_calibration_batches):
    """
    1. Compute movement/importance scores on calibration data
    2. Find global threshold at target sparsity
    3. Zero out weights below threshold (prune)
    """
    pruned = copy.deepcopy(model)

    # Step 1: compute scores
    print(f"    Computing movement scores on {n_calibration_batches} batches...")
    scores = compute_movement_scores(pruned, loader, n_calibration_batches)

    if not scores:
        print("    WARNING: No gradients computed. Falling back to L1 magnitude.")
        import torch.nn.utils.prune as prune_utils
        layers = [(m, "weight") for _, m in pruned.named_modules()
                  if isinstance(m, (nn.Conv2d, nn.Linear))]
        prune_utils.global_unstructured(layers,
                                        pruning_method=prune_utils.L1Unstructured,
                                        amount=sparsity)
        for module, param in layers:
            prune_utils.remove(module, param)
        return pruned

    # Step 2: find global threshold
    all_scores = torch.cat([s.view(-1) for s in scores.values()])
    k = max(1, int(all_scores.numel() * sparsity))
    threshold  = torch.kthvalue(all_scores, k).values.item()

    # Step 3: apply mask
    with torch.no_grad():
        for name, module in pruned.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)) and name in scores:
                score_tensor = scores[name].to(DEVICE)
                mask = (score_tensor > threshold).float()
                module.weight.data *= mask

    return pruned


def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000

def compute_flops(model):
    """Total theoretical FLOPs — shape-based, no weight values needed."""
    flop_counts = {}
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(module, nn.Conv2d):
                _, c_out, oH, oW = out.shape
                flop_counts[name] = 2 * module.in_channels * c_out * module.kernel_size[0] * module.kernel_size[1] * oH * oW
            elif isinstance(module, nn.Linear):
                flop_counts[name] = 2 * module.in_features * module.out_features
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        model(torch.randn(1, 3, 32, 32).to(DEVICE))

    for h in hooks: h.remove()
    return sum(flop_counts.values()), flop_counts

def compute_active_flops(model, flop_counts):
    """FLOPs scaled by per-layer weight density (approximates sparse compute)."""
    active = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in flop_counts:
            density = (module.weight != 0).float().mean().item()
            active += flop_counts[name] * density
    return int(active)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  4. Movement Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Scoring: Taylor |w*grad|")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()
    base_disk = disk_size_mb(model)
    base_flops, flop_counts = compute_flops(model)
    
    results    = []
    best_model = None
    best_score = -1.0
    best_sp    = None

    for sparsity in SPARSITY_LEVELS:
        print(f"\n  Sparsity {int(sparsity*100)}%...")
        pruned    = apply_movement_pruning(model, loader, sparsity, CALIBRATION_BATCHES)
        actual_sp = real_sparsity(pruned)
        total_p, active_p = count_params(pruned)
        metrics   = evaluate(pruned, loader)
        inf_gpu   = measure_gpu_ms(pruned)
        inf_cpu   = measure_cpu_ms(pruned)
        acc_drop  = baseline["accuracy"] - metrics["accuracy"]
        sp_mb     = sparse_size_mb(pruned)
        dk_mb     = disk_size_mb(pruned)

        row = {
            "target_sparsity"            : sparsity,
            "actual_sparsity"            : round(actual_sp, 4),
            "accuracy"                   : round(metrics["accuracy"], 6),
            "precision"                  : round(metrics["precision"], 6),
            "recall"                     : round(metrics["recall"], 6),
            "f1"                         : round(metrics["f1"], 6),
            "accuracy_drop"              : round(acc_drop, 6),
            "params_total"               : total_p,
            "params_active"              : active_p,
            "size_sparse_theoretical_mb" : round(sp_mb, 4),
            "size_disk_mb"               : round(dk_mb, 4),
            "disk_saved_mb"              : round(base_disk - dk_mb, 4),
            "inference_gpu_ms"           : round(inf_gpu, 4) if inf_gpu > 0 else None,
            "inference_cpu_ms"           : round(inf_cpu, 4),
            "calibration_batches_used"   : CALIBRATION_BATCHES,
            "flops_total"   : int(base_flops),
            "flops_active"  : compute_active_flops(pruned, flop_counts),
            "flops_reduction_pct": round((1 - compute_active_flops(pruned, flop_counts) / base_flops) * 100, 2)
        }
        results.append(row)
        print(f"  Actual sp: {actual_sp*100:.1f}% | Acc: {metrics['accuracy']:.4f} "
              f"(drop: {acc_drop:+.4f})")

        if acc_drop <= MAX_ACC_DROP:
            score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
            if score > best_score:
                best_score, best_model, best_sp = score, copy.deepcopy(pruned), sparsity

    report = {
        "method"              : "movement_pruning",
        "description"         : "Movement pruning via Taylor importance |w*grad| — data-aware pruning",
        "pruning_granularity" : "weight (unstructured)",
        "scoring_method"      : "Taylor first-order: |weight * gradient|",
        "calibration_batches" : CALIBRATION_BATCHES,
        "baseline"            : baseline,
        "device"              : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_sparsity"       : best_sp,
        "results"             : results,
        "notes": {
            "theory"          : "True movement pruning trains soft masks alongside weights. Here we use Taylor importance (|w*grad|) as a one-shot post-training approximation.",
            "advantage_over_magnitude": "Data-aware — considers gradient signal, not just weight value. Small weights with large gradients are preserved.",
            "limitation"      : "One-shot Taylor approximation; full movement pruning requires training with soft masks (see Sanh et al. 2020)",
        }
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")


if __name__ == "__main__":
    main()



  4. Movement Pruning — ResNet-50 / CIFAR-10
  Device: cuda  |  Scoring: Taylor |w*grad|


  Sparsity 30%...
    Computing movement scores on 10 batches...
  Actual sp: 32.1% | Acc: 0.9280 (drop: +0.0040)

  ✓ Saved -> 4_v2_movement_Pruning.json
